# Building an Interactive Echoshader App

This notebook demonstrates the `echoshader.app` layer.

The visualization functions themselves remain reusable elements. Panel widgets and HoloViews streams are then used to connect those elements into an interactive application.

Here, a box selection on an echogram updates both the corresponding vessel track and the `Sv` histogram for the selected data.


## Import packages


In [ ]:
from urllib import request

import holoviews as hv
import panel as pn
import xarray as xr

import echoshader
from echoshader.app import get_box_plot, get_box_stream

hv.extension("bokeh")
pn.extension()


## Load example data

The example uses processed Pacific hake survey data containing acoustic backscatter, ping time, latitude, and longitude.


In [ ]:
url = (
    "https://drive.google.com/uc?export=download"
    "&id=197D0MW-bHaF6mZLcQwyr4zqyEHIfwsep"
)

request.urlretrieve(
    url,
    "concatenated_MVBS.nc",
)

MVBS_ds = xr.open_mfdataset(
    paths="concatenated_MVBS.nc",
    data_vars="minimal",
    coords="minimal",
    combine="by_coords",
)

MVBS_ds


## Build the selectable echogram

The echogram is a normal Echoshader element. The box-selection stream and its visual overlay are added separately from `echoshader.app`.


In [ ]:
channel = MVBS_ds.channel.values[0]

echogram = (
    MVBS_ds.eshader.echogram(
        channel=channel,
    )
    .opts(
        cmap="jet",
        clim=(-80, -30),
        colorbar=True,
        tools=[
            "box_select",
            "hover",
        ],
        width=1000,
        height=400,
    )
)

box_stream = get_box_stream(
    echogram
)

box_overlay = get_box_plot(
    box_stream
)

echogram * box_overlay


## Convert the box selection back to xarray

The stream stores the selected plot bounds. The application layer converts those bounds into an xarray subset that can be reused by other elements.


In [ ]:
def data_from_box(
    ds,
    bounds,
    vert_dim="echo_range",
):
    if bounds is None:
        return ds

    left, bottom, right, top = bounds

    return ds.sel(
        ping_time=slice(
            left,
            right,
        ),
        **{
            vert_dim: slice(
                min(bottom, top),
                max(bottom, top),
            )
        },
    )


## Add application controls

These widgets control the dependent views. Their state is explicit in the notebook rather than stored on the xarray accessor.


In [ ]:
tile_select = pn.widgets.Select(
    name="Map tile",
    options=[
        "OpenTopoMap",
        "OSM",
    ],
    value="OpenTopoMap",
)

hist_bins = pn.widgets.IntInput(
    name="Histogram bins",
    value=24,
    start=1,
)

hist_overlay = pn.widgets.Checkbox(
    name="Overlay channels",
    value=True,
)


## Link the selection to the track and histogram

Both outputs depend on the same selection stream. Draw a box on the echogram and the geographic track and histogram update together.


In [ ]:
@pn.depends(
    box_stream.param.bounds,
    tile_select.param.value,
)
def selected_track(
    bounds,
    tile,
):
    selected_ds = data_from_box(
        MVBS_ds,
        bounds,
    )

    return (
        selected_ds.eshader.track(
            tile=tile,
        )
        .opts(
            width=700,
            height=400,
        )
    )


@pn.depends(
    box_stream.param.bounds,
    hist_bins.param.value,
    hist_overlay.param.value,
)
def selected_histogram(
    bounds,
    bins,
    overlay,
):
    selected_ds = data_from_box(
        MVBS_ds,
        bounds,
    )

    return (
        selected_ds.eshader.hist(
            bins=bins,
            overlay=overlay,
        )
        .opts(
            width=700,
            height=400,
        )
    )


## Compose the application


In [ ]:
controls = pn.Row(
    tile_select,
    hist_bins,
    hist_overlay,
)

app = pn.Column(
    controls,
    echogram * box_overlay,
    pn.Row(
        selected_track,
        selected_histogram,
    ),
)

app


The separation is now explicit:

- `elements/` creates the echogram, track, and histogram visualizations.
- `app/` provides interaction helpers such as the box-selection stream.
- Panel composes the elements and reactive state into an application.


## Serve the application

The same layout can be served directly with Panel:


In [ ]:
app.servable()


From the repository root:

```bash
panel serve notebooks/app_example.ipynb --show
```

The notebook also works normally in Jupyter or VS Code.
